In [1]:
import os
import pandas as pd
from pathlib import Path
import sys

BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed" 
PROCESSED_FEATURE_DIR = PROCESSED_DIR / "features"

COMPANIES_DIR = BASE_DIR / "data" / "raw" / "companies"

sys.path.append(str(BASE_DIR))

returns_df = pd.read_parquet(os.path.join(COMPANIES_DIR, "returns.parquet"))

feature_matrix_pipeline_dict = {
    "feature_matrix_stock": os.path.join(PROCESSED_FEATURE_DIR, "feature_matrix_stock.parquet"), 
    "feature_matrix_market": os.path.join(PROCESSED_FEATURE_DIR, "feature_matrix_market.parquet"), 
    "feature_matrix_macro_market": os.path.join(PROCESSED_FEATURE_DIR, "feature_matrix_macro_market.parquet")
}

In [2]:
from scripts.models.lr.lr_model import LinearRegressionModel

from scripts.models.dt.tune_dt import DTTuner
from scripts.models.dt.dt_model import DTRegressionModel

from scripts.models.lightgbm.tune_lightgbm import LightGBMTuner
from scripts.models.lightgbm.lightgbm_model import LightGBMRegressionModel

from scripts.models.xgboost.tune_xgboost import XGBoostTuner
from scripts.models.xgboost.xgboost_model import XGBoostRegressionModel
from scripts.models.walk_forward import WalkForwardValidator

from scripts.models.gru.tune_gru import GRUTuner
from scripts.models.gru.gru_model import GRURegressionModel
from scripts.models.walk_forward_gru import WalkForwardGRUValidator

I0000 00:00:1784217439.435247     600 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784217439.992989     600 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784217442.270011     600 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
feature_matrix_stock = pd.read_parquet(feature_matrix_pipeline_dict["feature_matrix_stock"])
feature_matrix_market = pd.read_parquet(feature_matrix_pipeline_dict["feature_matrix_market"])
feature_matrix_macro_market = pd.read_parquet(feature_matrix_pipeline_dict["feature_matrix_macro_market"])

In [10]:
feature_matrix_unscaled = feature_matrix_stock.columns

market_scaled_columns = feature_matrix_market.columns.difference(feature_matrix_unscaled).tolist()
macro_market_scaled_columns = feature_matrix_macro_market.columns.difference(feature_matrix_unscaled).tolist()

In [19]:
feature_matrix_unscaled

Index(['Date', 'Ticker', 'autocorr_21', 'autocorr_63', 'market_beta_63',
       'industry_beta_63', 'market_resid_vol_126', 'industry_resid_vol_126',
       'mean_volatility_10', 'mean_volatility_21', 'mean_volatility_63',
       'amihud_21', 'momentum_252_21', 'momentum_liquidity_21', 'reversal_5',
       'rsr_21', 'trend_21', 'trend_63', 'trend_126', 'r2_21', 'r2_63',
       'r2_126', 'future_return_5d'],
      dtype='object')

In [4]:
feature_matrix_market.columns

Index(['Date', 'Ticker', 'autocorr_21', 'autocorr_63', 'market_beta_63',
       'industry_beta_63', 'market_resid_vol_126', 'industry_resid_vol_126',
       'mean_volatility_10', 'mean_volatility_21', 'mean_volatility_63',
       'amihud_21', 'momentum_252_21_x', 'momentum_liquidity_21', 'reversal_5',
       'rsr_21', 'trend_21', 'trend_63', 'trend_126', 'r2_21', 'r2_63',
       'r2_126', 'future_return_5d', 'market_return_1d', 'market_return_5d',
       'market_return_21d', 'momentum_63_21', 'momentum_252_21_y',
       'market_volatility_21d', 'market_volatility_63d', 'raw_garch_vol',
       'market_drawdown_21d', 'market_drawdown_63d', 'market_drawdown_252d'],
      dtype='object')

In [4]:
pickle_model_stock_paths = { 
    "gru_model_stock": os.path.join(BASE_DIR, "results/gru_model/gru_model_stock.pkl"),
    "dt_model_stock": os.path.join(BASE_DIR, "results/dt_model/dt_model_stock.pkl"),
    "lightgbm_model_stock": os.path.join(BASE_DIR, "results/lightgbm_model/lightgbm_model_stock.pkl"), 
    "xgboost_model_stock": os.path.join(BASE_DIR, "results/xgboost_model/xgboost_model_stock.pkl")
}

pickle_model_market_paths = { 
    "gru_model_market": os.path.join(BASE_DIR, "results/gru_model/gru_model_market.pkl"),
    "dt_model_market": os.path.join(BASE_DIR, "results/dt_model/dt_model_market.pkl"),
    "lightgbm_model_market": os.path.join(BASE_DIR, "results/lightgbm_model/lightgbm_model_market.pkl"), 
    "xgboost_model_market": os.path.join(BASE_DIR, "results/xgboost_model/xgboost_model_market.pkl")
}

pickle_model_macro_market_paths = {
    "gru_model_macro_market": os.path.join(BASE_DIR, "results/gru_model/gru_model_macro_market.pkl"),
    "dt_model_macro_market": os.path.join(BASE_DIR, "results/dt_model/dt_model_macro_market.pkl"),
    "lightgbm_model_macro_market": os.path.join(BASE_DIR, "results/lightgbm_model/lightgbm_model_macro_market.pkl"), 
    "xgboost_model_macro_market": os.path.join(BASE_DIR, "results/xgboost_model/xgboost_model_macro_market.pkl")
}

In [5]:
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
import json
import pickle

### Company Wide Feature Matrix

In [17]:
lr = LinearRegressionModel()

wf = WalkForwardValidator(
    feature_matrix=feature_matrix_macro_market,
    model=lr,
    validation_start="2025-07-01",
    validation_end="2026-12-31",
    rebalance_date=1,
    min_train_size=25000,
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    ),
    include_groups = False
)

mean_ic = daily_ic.dropna().mean()

In [18]:
mean_ic

np.float64(0.047419945350007)

In [6]:
DTTuner(20, 42, feature_matrix_stock, "stock").run_data()

1/20: IC = 0.02101
2/20: IC = 0.02424
3/20: IC = 0.01723
4/20: IC = 0.01296
5/20: IC = 0.00955
6/20: IC = 0.00093
7/20: IC = 0.02379
8/20: IC = 0.02314
9/20: IC = 0.02138
10/20: IC = 0.01986
11/20: IC = 0.02444
12/20: IC = 0.03874
13/20: IC = 0.00652
14/20: IC = 0.04135
15/20: IC = 0.01360
16/20: IC = 0.01850
17/20: IC = 0.01737
18/20: IC = 0.02336
19/20: IC = 0.01028
20/20: IC = 0.02769


In [ ]:
results_dt_stock_df = pd.read_csv(os.path.join(BASE_DIR, "results/dt_model/random_search_stock.csv"))
results_dt_stock_df.head()

with open(os.path.join(BASE_DIR, "results/dt_model/best_params_stock.json"), "r") as f:
    best_dt_stock_params = json.load(f)
    
int_dt_params = [
    "max_depth",
    "min_samples_split",
    "min_samples_leaf"
]

for p in int_dt_params:
    best_dt_stock_params[p] = int(best_dt_stock_params[p])
    
dt_stock_model = LightGBMRegressionModel(params=best_dt_stock_params)

with open(pickle_model_stock_paths["dt_model_stock"], "wb") as file: 
    pickle.dump(dt_stock_model, file)
    
with open(pickle_model_stock_paths["dt_model_stock"], "rb") as file: 
    dt_stock_v2 = pickle.load(file)
    
wf = WalkForwardValidator(
    feature_matrix=feature_matrix_stock,
    model=dt_stock_v2,
    validation_start="2025-07-01",
    validation_end="2026-12-31",
    rebalance_date=1,
    min_train_size=25000,
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    ),
    include_groups = False
)

mean_ic = daily_ic.dropna().mean()

In [7]:
results_dt_stock_df = pd.read_csv(os.path.join(BASE_DIR, "results/dt_model/random_search_stock.csv"))
results_dt_stock_df.head()

,max_depth,min_samples_split,min_samples_leaf,max_features,mean_ic
0,5,100,10,0.5,0.041347
1,5,20,20,0.5,0.038738
2,4,20,20,0.5,0.027695
3,3,50,10,NaN,0.024445
4,3,20,10,0.8,0.024241


In [8]:
with open(os.path.join(BASE_DIR, "results/dt_model/best_params_stock.json"), "r") as f:
    best_dt_stock_params = json.load(f)

In [10]:
int_dt_params = [
    "max_depth",
    "min_samples_split",
    "min_samples_leaf"
]

for p in int_dt_params:
    best_dt_stock_params[p] = int(best_dt_stock_params[p])

In [11]:
dt_stock_model = LightGBMRegressionModel(params=best_dt_stock_params)

with open(pickle_model_stock_paths["dt_model_stock"], "wb") as file: 
    pickle.dump(dt_stock_model, file)

In [12]:
with open(pickle_model_stock_paths["dt_model_stock"], "rb") as file: 
    dt_stock_v2 = pickle.load(file)

In [13]:
wf = WalkForwardValidator(
    feature_matrix=feature_matrix_stock,
    model=dt_stock_v2,
    validation_start="2025-07-01",
    validation_end="2026-12-31",
    rebalance_date=1,
    min_train_size=25000,
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    ),
    include_groups = False
)

mean_ic = daily_ic.dropna().mean()

In [14]:
mean_ic

np.float64(0.027982726050651403)

#### LightGBM

In [6]:
LightGBMTuner(20, 42, feature_matrix_stock, "stock").run_data()

1/20: IC = 0.02809
2/20: IC = 0.03027
3/20: IC = 0.03006
4/20: IC = 0.01855
5/20: IC = 0.02759
6/20: IC = 0.03432
7/20: IC = 0.02748
8/20: IC = 0.03119
9/20: IC = 0.01819
10/20: IC = 0.01745
11/20: IC = 0.03407
12/20: IC = 0.01409
13/20: IC = 0.01656
14/20: IC = 0.00390
15/20: IC = 0.02210
16/20: IC = 0.03091
17/20: IC = 0.02585
18/20: IC = 0.01486
19/20: IC = 0.02583
20/20: IC = 0.02404


In [7]:
results_lightgbm_stock_df = pd.read_csv(os.path.join(BASE_DIR, "results/lightgbm_model/random_search_stock.csv"))
results_lightgbm_stock_df.head()

,learning_rate,num_leaves,max_depth,n_estimators,min_child_samples,mean_ic
0,0.08,63,9,200,20,0.034324
1,0.02,63,5,200,10,0.034073
2,0.04,63,9,100,100,0.031193
3,0.01,31,5,300,100,0.030914
4,0.02,127,5,100,100,0.030269


In [ ]:
with open(os.path.join(BASE_DIR, "results/lightgbm_model/best_params_stock.json"), "r") as f:
    best_lightgbm_stock_params = json.load(f)

In [38]:
int_lightgbm_params = [
    "num_leaves",
    "max_depth",
    "n_estimators",
    "min_child_samples",
]

for p in int_lightgbm_params:
    best_lightgbm_stock_params[p] = int(best_lightgbm_stock_params[p])

In [24]:
lightgbm_stock_model = LightGBMRegressionModel(params=best_lightgbm_stock_params)

with open(pickle_model_stock_paths["lightgbm_model_stock"], "wb") as file: 
    pickle.dump(lightgbm_stock_model, file)

In [25]:
with open(pickle_model_stock_paths["lightgbm_model_stock"], "rb") as file: 
    lightgbm_stock_v2 = pickle.load(file)

In [28]:
wf = WalkForwardValidator(
    feature_matrix=feature_matrix_stock,
    model=lightgbm_stock_v2,
    validation_start="2025-07-01",
    validation_end="2026-12-31",
    rebalance_date=1,
    min_train_size=25000,
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    ),
    include_groups = False
)

mean_ic = daily_ic.dropna().mean()

In [32]:
mean_ic.astype("float32")

np.float32(0.020762276)

#### XGBoost

In [33]:
XGBoostTuner(20, 42, feature_matrix_stock, "stock").run_data()

1/20: IC = 0.01632
2/20: IC = 0.03080
3/20: IC = 0.02464
4/20: IC = 0.01850
5/20: IC = 0.02925
6/20: IC = 0.02273
7/20: IC = 0.00860
8/20: IC = 0.01570
9/20: IC = 0.03015
10/20: IC = 0.03154
11/20: IC = 0.02845
12/20: IC = 0.00879
13/20: IC = 0.02795
14/20: IC = 0.02717
15/20: IC = 0.01741
16/20: IC = 0.01432
17/20: IC = 0.02059
18/20: IC = 0.02791
19/20: IC = 0.01429
20/20: IC = 0.02044


In [35]:
results_xgboost_stock_df = pd.read_csv(os.path.join(BASE_DIR, "results/xgboost_model/random_searchstock.csv"))
results_xgboost_stock_df.head()

,learning_rate,max_depth,n_estimators,min_child_weight,mean_ic
0,0.01,3,500,10,0.031537
1,0.02,4,200,10,0.030796
2,0.04,4,200,50,0.030146
3,0.02,7,100,20,0.029247
4,0.04,5,300,10,0.028449


In [40]:
with open(os.path.join(BASE_DIR, "results/xgboost_model/best_params_stock.json"), "r") as f:
    best_xgboost_stock_params = json.load(f)

In [42]:
int_xgboost_params = [
    "max_depth",
    "n_estimators",
    "min_child_weight"
]

for p in int_xgboost_params:
    best_xgboost_stock_params[p] = int(best_xgboost_stock_params[p])

In [43]:
xgboost_stock_model = XGBoostRegressionModel(params=best_xgboost_stock_params)

with open(pickle_model_stock_paths["xgboost_model_stock"], "wb") as file: 
    pickle.dump(xgboost_stock_model, file)

In [44]:
with open(pickle_model_stock_paths["xgboost_model_stock"], "rb") as file: 
    xgboost_stock_v2 = pickle.load(file)

In [45]:
wf = WalkForwardValidator(
    feature_matrix=feature_matrix_stock,
    model=xgboost_stock_v2,
    validation_start="2025-07-01",
    validation_end="2026-12-31",
    rebalance_date=1,
    min_train_size=25000,
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    ),
    include_groups = False
)

mean_ic = daily_ic.dropna().mean()

In [46]:
mean_ic

np.float64(0.04539480966151561)

#### Decision Tree

In [18]:
DTTuner(20, 123, feature_matrix_market, "market").run_data()

/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


1/20: IC = 0.04503


/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


2/20: IC = -0.01880


/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


...

/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


18/20: IC = 0.00667


/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


19/20: IC = 0.00011
20/20: IC = -0.01785


/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


In [19]:
results_dt_market_df = pd.read_csv(os.path.join(BASE_DIR, "results/dt_model/random_search_market.csv"))
results_dt_market_df.head()

,max_depth,min_samples_split,min_samples_leaf,max_features,mean_ic
0,2,50,10,0.5,0.045031
1,2,20,10,0.5,0.045031
2,6,50,20,0.5,0.031889
3,6,100,20,NaN,0.030488
4,6,50,50,NaN,0.030304


In [21]:
with open(os.path.join(BASE_DIR, "results/dt_model/best_params_market.json"), "r") as f:
    best_dt_market_params = json.load(f)

In [22]:
int_dt_params = [
    "max_depth",
    "min_samples_split",
    "min_samples_leaf"
]

for p in int_dt_params:
    best_dt_market_params[p] = int(best_dt_market_params[p])

In [31]:
dt_market_model = DTRegressionModel(params=best_dt_market_params)

with open(pickle_model_market_paths["dt_model_market"], "wb") as file: 
    pickle.dump(dt_market_model, file)

In [33]:
with open(pickle_model_market_paths["dt_model_market"], "rb") as file: 
    dt_market_v2 = pickle.load(file)

In [34]:
wf = WalkForwardValidator(
    feature_matrix=feature_matrix_market,
    model=dt_market_v2,
    validation_start="2025-07-01",
    validation_end="2026-12-31",
    rebalance_date=1,
    min_train_size=25000,
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    ),
    include_groups = False
)

mean_ic = daily_ic.dropna().mean()

/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


In [36]:
mean_ic

np.float64(0.03680510430828885)

In [7]:
LightGBMTuner(20, 42, feature_matrix_market, "market").run_data()

1/20: IC = 0.02249
2/20: IC = 0.01671
3/20: IC = 0.01870
4/20: IC = 0.01123
5/20: IC = 0.03230
6/20: IC = -0.00142
7/20: IC = 0.01079
8/20: IC = 0.02340
9/20: IC = -0.00420
10/20: IC = 0.00152
11/20: IC = 0.02322
12/20: IC = -0.00587
13/20: IC = 0.00578
14/20: IC = -0.00160
15/20: IC = 0.00499
16/20: IC = 0.02379
17/20: IC = -0.00287
18/20: IC = 0.00908
19/20: IC = -0.00085
20/20: IC = -0.00050


In [8]:
results_lightgbm_market_df = pd.read_csv(os.path.join(BASE_DIR, "results/lightgbm_model/random_search_market.csv"))
results_lightgbm_market_df.head()

,learning_rate,num_leaves,max_depth,n_estimators,min_child_samples,mean_ic
0,0.02,63,9,100,20,0.032295
1,0.01,31,5,300,100,0.023795
2,0.04,63,9,100,100,0.023400
3,0.02,63,5,200,10,0.023225
4,0.01,31,9,200,20,0.022490


In [10]:
with open(os.path.join(BASE_DIR, "results/lightgbm_model/best_params_market.json"), "r") as f:
    best_lightgbm_market_params = json.load(f)

In [11]:
int_lightgbm_params = [
    "num_leaves",
    "max_depth",
    "n_estimators",
    "min_child_samples",
]

for p in int_lightgbm_params:
    best_lightgbm_market_params[p] = int(best_lightgbm_market_params[p])

In [13]:
lightgbm_market_model = LightGBMRegressionModel(params=best_lightgbm_market_params)

with open(pickle_model_market_paths["lightgbm_model_market"], "wb") as file: 
    pickle.dump(lightgbm_market_model, file)

In [16]:
with open(pickle_model_market_paths["lightgbm_model_market"], "rb") as file: 
    lightgbm_market_v2 = pickle.load(file)

In [17]:
wf = WalkForwardValidator(
    feature_matrix=feature_matrix_market,
    model=lightgbm_market_v2,
    validation_start="2025-07-01",
    validation_end="2026-12-31",
    rebalance_date=1,
    min_train_size=25000,
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    ),
    include_groups = False
)

mean_ic = daily_ic.dropna().mean()

In [18]:
mean_ic

np.float64(0.017541027526196136)

In [21]:
XGBoostTuner(20, 42, feature_matrix_market, "market").run_data()

1/20: IC = 0.01563
2/20: IC = 0.01879
3/20: IC = 0.00135


/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


4/20: IC = 0.01874
5/20: IC = 0.01256
6/20: IC = 0.00524
7/20: IC = -0.00381
8/20: IC = -0.01135
9/20: IC = 0.01049
10/20: IC = 0.02470
11/20: IC = 0.00067
12/20: IC = 0.01368
13/20: IC = 0.01004
14/20: IC = 0.00376
15/20: IC = 0.00328
16/20: IC = 0.00454
17/20: IC = 0.00748
18/20: IC = 0.01195
19/20: IC = 0.01074
20/20: IC = 0.01308


In [25]:
results_xgboost_market_df = pd.read_csv(os.path.join(BASE_DIR, "results/xgboost_model/random_searchmarket.csv"))
results_xgboost_market_df.head()

,learning_rate,max_depth,n_estimators,min_child_weight,mean_ic
0,0.01,3,500,10,0.024702
1,0.02,4,200,10,0.018790
2,0.01,3,100,20,0.018744
3,0.20,3,100,50,0.015628
4,0.20,6,100,100,0.013683


In [26]:
with open(os.path.join(BASE_DIR, "results/xgboost_model/best_params_market.json"), "r") as f:
    best_xgboost_market_params = json.load(f)

In [27]:
int_xgboost_params = [
    "max_depth",
    "n_estimators",
    "min_child_weight"
]

for p in int_xgboost_params:
    best_xgboost_market_params[p] = int(best_xgboost_market_params[p])

In [29]:
xgboost_market_model = XGBoostRegressionModel(params=best_xgboost_market_params)

with open(pickle_model_market_paths["xgboost_model_market"], "wb") as file: 
    pickle.dump(xgboost_market_model, file)

In [30]:
with open(pickle_model_market_paths["xgboost_model_market"], "rb") as file: 
    xgboost_market_v2 = pickle.load(file)

In [31]:
wf = WalkForwardValidator(
    feature_matrix=feature_matrix_market,
    model=xgboost_market_v2,
    validation_start="2025-07-01",
    validation_end="2026-12-31",
    rebalance_date=1,
    min_train_size=25000,
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    ),
    include_groups = False
)

mean_ic = daily_ic.dropna().mean()

In [32]:
mean_ic

np.float64(0.05022851288994176)

DTTuner(20, 123, feature_matrix_market, "market").run_data()

In [28]:
DTTuner(20, 123, feature_matrix_macro_market, "macro_market").run_data()

/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


1/20: IC = -0.03324


/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


2/20: IC = -0.03457


/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


...

/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


18/20: IC = -0.05167


/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


19/20: IC = -0.07949
20/20: IC = -0.03536


/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


In [29]:
results_dt_macro_market_df = pd.read_csv(os.path.join(BASE_DIR, "results/dt_model/random_search_macro_market.csv"))
results_dt_macro_market_df.head()

,max_depth,min_samples_split,min_samples_leaf,max_features,mean_ic
0,6,100,20,0.5,0.012149
1,6,50,20,0.5,0.005963
2,5,100,50,0.5,-0.001580
3,5,100,50,0.5,-0.001580
4,6,100,20,NaN,-0.004089


In [30]:
with open(os.path.join(BASE_DIR, "results/dt_model/best_params_macro_market.json"), "r") as f:
    best_dt_macro_market_params = json.load(f)

In [40]:
int_dt_params = [
    "max_depth",
    "min_samples_split",
    "min_samples_leaf"
]

for p in int_dt_params:
    best_dt_macro_market_params[p] = int(best_dt_macro_market_params[p])

In [44]:
dt_macro_market_model = DTRegressionModel(params=best_dt_macro_market_params)

with open(pickle_model_macro_market_paths["dt_model_macro_market"], "wb") as file: 
    pickle.dump(dt_macro_market_model, file)

In [45]:
with open(pickle_model_macro_market_paths["dt_model_macro_market"], "rb") as file: 
    dt_macro_market_v2 = pickle.load(file)

In [46]:
wf = WalkForwardValidator(
    feature_matrix=feature_matrix_macro_market,
    model=dt_macro_market_v2,
    validation_start="2025-07-01",
    validation_end="2026-12-31",
    rebalance_date=1,
    min_train_size=25000,
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    ),
    include_groups = False
)

mean_ic = daily_ic.dropna().mean()

/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


In [47]:
mean_ic

np.float64(0.019320281953072903)

#### LightGBM

In [20]:
LightGBMTuner(20, 42, feature_matrix_macro_market, "macro_market").run_data()

1/20: IC = -0.00853
2/20: IC = 0.00403
3/20: IC = -0.00470
4/20: IC = -0.00893
5/20: IC = -0.02606
6/20: IC = -0.01282
7/20: IC = -0.00303
8/20: IC = -0.00987
9/20: IC = -0.00954
10/20: IC = -0.01549
11/20: IC = -0.01249
12/20: IC = -0.00755
13/20: IC = -0.01469
14/20: IC = -0.00308
15/20: IC = -0.01732
16/20: IC = -0.00942
17/20: IC = -0.00974
18/20: IC = -0.01244
19/20: IC = -0.01017
20/20: IC = 0.00313


In [33]:
results_lightgbm_macro_market_df = pd.read_csv(os.path.join(BASE_DIR, "results/lightgbm_model/random_search_macro_market.csv"))
results_lightgbm_macro_market_df.head()

,learning_rate,num_leaves,max_depth,n_estimators,min_child_samples,mean_ic
0,0.02,127,5,100,100,0.004035
1,0.04,31,7,500,10,0.003135
2,0.04,31,5,500,10,-0.003029
3,0.10,127,7,200,20,-0.003081
4,0.01,31,5,200,20,-0.004695


In [34]:
with open(os.path.join(BASE_DIR, "results/lightgbm_model/best_params_macro_market.json"), "r") as f:
    best_lightgbm_macro_market_params = json.load(f)

In [35]:
int_lightgbm_params = [
    "num_leaves",
    "max_depth",
    "n_estimators",
    "min_child_samples",
]

for p in int_lightgbm_params:
    best_lightgbm_macro_market_params[p] = int(best_lightgbm_macro_market_params[p])

In [39]:
lightgbm_macro_market_model = LightGBMRegressionModel(params=best_lightgbm_macro_market_params)

with open(pickle_model_macro_market_paths["lightgbm_model_macro_market"], "wb") as file: 
    pickle.dump(lightgbm_macro_market_model, file)

In [40]:
with open(pickle_model_macro_market_paths["lightgbm_model_macro_market"], "rb") as file: 
    lightgbm_macro_market_v2 = pickle.load(file)

In [41]:
wf = WalkForwardValidator(
    feature_matrix=feature_matrix_macro_market,
    model=lightgbm_macro_market_v2,
    validation_start="2025-07-01",
    validation_end="2026-12-31",
    rebalance_date=1,
    min_train_size=25000,
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    ),
    include_groups = False
)

mean_ic = daily_ic.dropna().mean()

In [42]:
mean_ic

np.float64(0.05662860632913941)

In [22]:
XGBoostTuner(20, 42, feature_matrix_macro_market, "macro_market").run_data()

1/20: IC = 0.01260
2/20: IC = -0.00618
3/20: IC = -0.01592


/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


4/20: IC = -0.00806
5/20: IC = -0.02053
6/20: IC = 0.00458
7/20: IC = -0.00391
8/20: IC = -0.01012
9/20: IC = -0.00763
10/20: IC = -0.00602
11/20: IC = -0.00055
12/20: IC = -0.00787
13/20: IC = -0.01575
14/20: IC = -0.00021
15/20: IC = -0.00984
16/20: IC = 0.00280
17/20: IC = -0.00521
18/20: IC = -0.00605
19/20: IC = 0.00033
20/20: IC = -0.00066


In [45]:
results_xgboost_macro_market_df = pd.read_csv(os.path.join(BASE_DIR, "results/xgboost_model/random_searchmacro_market.csv"))
results_xgboost_macro_market_df.head()

,learning_rate,max_depth,n_estimators,min_child_weight,mean_ic
0,0.2,3,100,50,0.012600
1,0.2,7,500,20,0.004576
2,0.3,4,100,100,0.002805
3,0.2,3,200,20,0.000329
4,0.1,4,100,10,-0.000213


In [46]:
with open(os.path.join(BASE_DIR, "results/xgboost_model/best_params_macro_market.json"), "r") as f:
    best_xgboost_macro_market_params = json.load(f)

In [47]:
int_xgboost_params = [
    "max_depth",
    "n_estimators",
    "min_child_weight"
]

for p in int_xgboost_params:
    best_xgboost_macro_market_params[p] = int(best_xgboost_macro_market_params[p])

In [48]:
xgboost_macro_market_model = XGBoostRegressionModel(params=best_xgboost_macro_market_params)

with open(pickle_model_macro_market_paths["xgboost_model_macro_market"], "wb") as file: 
    pickle.dump(xgboost_macro_market_model, file)

In [49]:
with open(pickle_model_macro_market_paths["xgboost_model_macro_market"], "rb") as file: 
    xgboost_macro_market_v2 = pickle.load(file)

In [50]:
wf = WalkForwardValidator(
    feature_matrix=feature_matrix_macro_market,
    model=xgboost_macro_market_v2,
    validation_start="2025-07-01",
    validation_end="2026-12-31",
    rebalance_date=1,
    min_train_size=25000,
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    ),
    include_groups = False
)

mean_ic = daily_ic.dropna().mean()

In [51]:
mean_ic

np.float64(0.03967915543911888)

#### Gated Recurrent Unit

In [5]:
GRUTuner(6, 42, feature_matrix_stock).run_data()

TensorFlow GPU: using /physical_device:GPU:0
GRU fold 1/103 2023-07-04: skipped (train=1880, test=188, reason=train_too_small)
GRU fold 2/103 2023-07-11: skipped (train=2820, test=188, reason=train_too_small)
GRU fold 3/103 2023-07-18: skipped (train=3760, test=188, reason=train_too_small)
GRU fold 4/103 2023-07-25: skipped (train=4700, test=188, reason=train_too_small)
GRU fold 5/103 2023-08-01: skipped (train=5640, test=189, reason=train_too_small)
GRU fold 6/103 2023-08-08: skipped (train=6581, test=189, reason=train_too_small)
GRU fold 7/103 2023-08-15: skipped (train=7526, test=189, reason=train_too_small)
GRU fold 8/103 2023-08-22: skipped (train=8471, test=189, reason=train_too_small)
GRU fold 9/103 2023-08-29: skipped (train=9416, test=189, reason=train_too_small)
GRU fold 10/103 2023-09-05: fresh training (train=10361, test=189)


I0000 00:00:1783429756.400885   20194 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5590 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1783429757.602859   20379 cuda_dnn.cc:461] Loaded cuDNN version 92400


ListNet epoch 1/10: loss = 5.244968
ListNet epoch 2/10: loss = 5.250819
ListNet epoch 3/10: loss = 5.268196
ListNet epoch 4/10: loss = 5.204166
ListNet epoch 5/10: loss = 5.168419
ListNet epoch 6/10: loss = 5.082496
ListNet epoch 7/10: loss = 5.089852
ListNet epoch 8/10: loss = 5.004738
ListNet epoch 9/10: loss = 4.943456
ListNet epoch 10/10: loss = 4.882118
GRU fold 11/103 2023-09-12: fine-tune training (train=11306, test=189)
ListNet epoch 1/2: loss = 4.903396
ListNet epoch 2/2: loss = 4.936039
GRU fold 12/103 2023-09-19: fine-tune training (train=12251, test=189)
ListNet epoch 1/2: loss = 4.806054
ListNet epoch 2/2: loss = 4.792139
GRU fold 13/103 2023-09-26: fine-tune training (train=13196, test=189)
ListNet epoch 1/2: loss = 4.758044
ListNet epoch 2/2: loss = 4.765497
GRU fold 14/103 2023-10-03: fine-tune training (train=14141, test=189)
ListNet epoch 1/2: loss = 4.849443
ListNet epoch 2/2: loss = 4.975499
GRU fold 15/103 2023-10-10: fine-tune training (train=15086, test=190)
List

/mnt/c/Users/Gordon Li/Desktop/systematic_project/scripts/models/gru/tune_gru.py:88: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_ic = preds.groupby("Date").apply(


1/6: IC = 0.02033 (raw_ic=0.02033, trained_folds=94, skipped_folds=9, coverage=91.3%)
GRU fold 1/103 2023-07-04: skipped (train=1880, test=188, reason=train_too_small)
GRU fold 2/103 2023-07-11: skipped (train=2820, test=188, reason=train_too_small)
GRU fold 3/103 2023-07-18: skipped (train=3760, test=188, reason=train_too_small)
GRU fold 4/103 2023-07-25: skipped (train=4700, test=188, reason=train_too_small)
GRU fold 5/103 2023-08-01: skipped (train=5640, test=189, reason=train_too_small)
GRU fold 6/103 2023-08-08: skipped (train=6581, test=189, reason=train_too_small)
GRU fold 7/103 2023-08-15: skipped (train=7526, test=189, reason=train_too_small)
GRU fold 8/103 2023-08-22: skipped (train=8471, test=189, reason=train_too_small)
GRU fold 9/103 2023-08-29: skipped (train=9416, test=189, reason=train_too_small)
GRU fold 10/103 2023-09-05: fresh training (train=10361, test=189)
ListNet epoch 1/10: loss = 5.245539
ListNet epoch 2/10: loss = 5.250664
ListNet epoch 3/10: loss = 5.276170
L

...

/mnt/c/Users/Gordon Li/Desktop/systematic_project/scripts/models/gru/tune_gru.py:88: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_ic = preds.groupby("Date").apply(


4/6: IC = 0.01674 (raw_ic=0.01674, trained_folds=94, skipped_folds=9, coverage=91.3%)
GRU fold 1/103 2023-07-04: skipped (train=1880, test=188, reason=train_too_small)
GRU fold 2/103 2023-07-11: skipped (train=2820, test=188, reason=train_too_small)
GRU fold 3/103 2023-07-18: skipped (train=3760, test=188, reason=train_too_small)
GRU fold 4/103 2023-07-25: skipped (train=4700, test=188, reason=train_too_small)
GRU fold 5/103 2023-08-01: skipped (train=5640, test=189, reason=train_too_small)
GRU fold 6/103 2023-08-08: skipped (train=6581, test=189, reason=train_too_small)
GRU fold 7/103 2023-08-15: skipped (train=7526, test=189, reason=train_too_small)
GRU fold 8/103 2023-08-22: skipped (train=8471, test=189, reason=train_too_small)
GRU fold 9/103 2023-08-29: skipped (train=9416, test=189, reason=train_too_small)
GRU fold 10/103 2023-09-05: fresh training (train=10361, test=189)
ListNet epoch 1/10: loss = 5.292879
ListNet epoch 2/10: loss = 5.219213
ListNet epoch 3/10: loss = 5.162806
L

/mnt/c/Users/Gordon Li/Desktop/systematic_project/scripts/models/gru/tune_gru.py:88: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_ic = preds.groupby("Date").apply(


5/6: IC = 0.01889 (raw_ic=0.01889, trained_folds=94, skipped_folds=9, coverage=91.3%)
GRU fold 1/103 2023-07-04: skipped (train=1880, test=188, reason=train_too_small)
GRU fold 2/103 2023-07-11: skipped (train=2820, test=188, reason=train_too_small)
GRU fold 3/103 2023-07-18: skipped (train=3760, test=188, reason=train_too_small)
GRU fold 4/103 2023-07-25: skipped (train=4700, test=188, reason=train_too_small)
GRU fold 5/103 2023-08-01: skipped (train=5640, test=189, reason=train_too_small)
GRU fold 6/103 2023-08-08: skipped (train=6581, test=189, reason=train_too_small)
GRU fold 7/103 2023-08-15: skipped (train=7526, test=189, reason=train_too_small)
GRU fold 8/103 2023-08-22: skipped (train=8471, test=189, reason=train_too_small)
GRU fold 9/103 2023-08-29: skipped (train=9416, test=189, reason=train_too_small)
GRU fold 10/103 2023-09-05: fresh training (train=10361, test=189)
ListNet epoch 1/10: loss = 5.299976
ListNet epoch 2/10: loss = 5.321143
ListNet epoch 3/10: loss = 5.324509
L

/mnt/c/Users/Gordon Li/Desktop/systematic_project/scripts/models/gru/tune_gru.py:88: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_ic = preds.groupby("Date").apply(


In [7]:
results_df = pd.read_csv(os.path.join(BASE_DIR, "results/gru_model/random_search.csv"))
results_df.head()

,dropout_rate,hidden_dim,learning_rate,epochs,fine_tune_epochs,early_stopping_patience,listnet_temperature,listnet_max_train_groups,sequence_length,min_train_size,...,raw_mean_ic,trained_folds,skipped_folds,coverage,skipped_train_too_small,skipped_empty_test,missing_strategy,allow_short_sequences,add_listing_age_features,precompute_sequences
0,0.1,32,0.0010,10,2,3,0.5,252,10,10000,...,0.020328,94,9,0.912621,9,0,ffill_zero_indicator,True,True,True
1,0.1,64,0.0005,10,2,3,0.5,252,10,10000,...,0.018892,94,9,0.912621,9,0,ffill_zero_indicator,True,True,True
2,0.2,32,0.0002,10,2,3,0.5,252,20,10000,...,0.017646,94,9,0.912621,9,0,ffill_zero_indicator,True,True,True
3,0.1,32,0.0010,10,3,3,0.5,252,20,10000,...,0.017309,94,9,0.912621,9,0,ffill_zero_indicator,True,True,True
4,0.1,64,0.0002,10,2,3,0.5,378,10,10000,...,0.016738,94,9,0.912621,9,0,ffill_zero_indicator,True,True,True


In [ ]:
results_df["output"]

In [10]:
import json
with open(os.path.join(BASE_DIR, "results/gru_model/best_params.json"), "r") as f:
    best_params = json.load(f)

In [14]:
best_params["output_dim"]

KeyError: 'output_dim'

In [20]:
model = GRURegressionModel(
    dropout_rate=best_params['dropout_rate'], 
    hidden_dim = best_params["hidden_dim"],
    output_dim=1
)

wf = WalkForwardGRUValidator(
    feature_matrix=feature_matrix_stock,
    model=model,
    validation_start="2025-07-01",
    validation_end="2026-06-30",
    rebalance_date=1,
    min_train_size=best_params.get("min_train_size", 10000),
    sequence_length=best_params.get("sequence_length", 10),

    training_mode="listnet",
    transfer_learning=True,
    reset_model_every_n_folds=26,

    missing_strategy="ffill_zero_indicator",
    allow_short_sequences=True,
    add_listing_age_features=True,
    precompute_sequences=True,
    verbose=1,

    fit_kwargs={
        "epochs": best_params.get("epochs", 10),
        "fine_tune_epochs": best_params.get("fine_tune_epochs", 2),
        "learning_rate": best_params.get("learning_rate", 0.001),
        "verbose": 1,
        "early_stopping_patience": best_params.get("early_stopping_patience", 3),
        "early_stopping_min_delta": 1e-4,
        "restore_best_weights": True,
        "listnet_target_transform": "zscore",
        "listnet_temperature": best_params.get("listnet_temperature", 0.5),
        "listnet_max_train_groups": best_params.get("listnet_max_train_groups", 252),
    },
    predict_kwargs={"verbose": 0},
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    ),
    include_groups = False
)

mean_ic = daily_ic.dropna().mean()

GRU fold 1/50 2025-07-01: fresh training (train=98864, test=197)
ListNet epoch 1/10: loss = 5.340822
ListNet epoch 2/10: loss = 5.307484
ListNet epoch 3/10: loss = 5.222168
ListNet epoch 4/10: loss = 5.300815
ListNet epoch 5/10: loss = 5.291175
ListNet epoch 6/10: loss = 5.287746
Early stopping at epoch 6/10
GRU fold 2/50 2025-07-08: fine-tune training (train=99849, test=197)
ListNet epoch 1/2: loss = 5.215633
ListNet epoch 2/2: loss = 5.206924
GRU fold 3/50 2025-07-15: fine-tune training (train=100834, test=197)
ListNet epoch 1/2: loss = 5.287105
ListNet epoch 2/2: loss = 5.323132
GRU fold 4/50 2025-07-22: fine-tune training (train=101819, test=197)
ListNet epoch 1/2: loss = 5.256243
ListNet epoch 2/2: loss = 5.256379
GRU fold 5/50 2025-07-29: fine-tune training (train=102804, test=197)
ListNet epoch 1/2: loss = 5.119458
ListNet epoch 2/2: loss = 5.149082
GRU fold 6/50 2025-08-05: fine-tune training (train=103789, test=197)
ListNet epoch 1/2: loss = 5.002885
ListNet epoch 2/2: loss = 

/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


In [25]:
daily_ic.isna().sum(), len(daily_ic)

(np.int64(1), 49)

In [24]:
daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(x["future_return_5d"], method="spearman"),
    include_groups=False,
)

daily_ic.mean()

/home/gordon1999/.local/lib/python3.10/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


np.float64(0.05578925339018039)

In [23]:
mean_ic

np.float64(0.05578925339018039)

In [11]:
best_params

{'dropout_rate': 0.1,
 'hidden_dim': 32,
 'learning_rate': 0.001,
 'epochs': 10,
 'fine_tune_epochs': 2,
 'early_stopping_patience': 3,
 'listnet_temperature': 0.5,
 'listnet_max_train_groups': 252,
 'sequence_length': 10,
 'min_train_size': 10000,
 'model_type': 'gru',
 'raw_mean_ic': 0.020327647815369766,
 'trained_folds': 94,
 'skipped_folds': 9,
 'coverage': 0.912621359223301,
 'skipped_train_too_small': 9,
 'skipped_empty_test': 0,
 'missing_strategy': 'ffill_zero_indicator',
 'allow_short_sequences': True,
 'add_listing_age_features': True,
 'precompute_sequences': True}

In [6]:
import json

In [ ]:
XGBoostTuner(20, 42, feature_matrix_stock).run_data()

In [ ]:
import json
with open(os.path.join(BASE_DIR, "results/xgboost_model/best_params.json"), "r") as f:
    best_params = json.load(f)
    
int_params = [
    "max_depth",
    "n_estimators",
    "min_child_weight",
]

for p in int_params:
    best_params[p] = int(best_params[p])

In [ ]:
best_params

In [ ]:
model = XGBoostRegressionModel(params=best_params)

wf = WalkForwardValidator(
    feature_matrix=feature_matrix_stock,
    model=model,
    validation_start="2025-07-01",
    validation_end="2026-12-31",
    rebalance_date=1,
    min_train_size=25000,
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    ),
    include_groups = False
)

mean_ic = daily_ic.dropna().mean()

In [ ]:
mean_ic

In [ ]:
LightGBMTuner(20, 42, feature_matrix_stock).run_data()

In [ ]:
results_df = pd.read_csv(os.path.join(BASE_DIR, "results/lightgbm_model/random_search.csv"))
results_df.head()

In [ ]:
import json
with open(os.path.join(BASE_DIR, "results/lightgbm_model/best_params.json"), "r") as f:
    best_params = json.load(f)

In [ ]:
int_params = [
    "num_leaves",
    "max_depth",
    "n_estimators",
    "min_child_samples",
]

for p in int_params:
    best_params[p] = int(best_params[p])

In [ ]:
model = LightGBMRegressionModel(params=best_params)

wf = WalkForwardValidator(
    feature_matrix=feature_matrix_market,
    model=model,
    validation_start="2025-07-01",
    validation_end="2026-12-31",
    rebalance_date=1,
    min_train_size=25000,
)

X_test, test_preds = wf.run_data()

daily_ic = test_preds.groupby("Date").apply(
    lambda x: x["prediction"].corr(
        x["future_return_5d"],
        method="spearman",
    )
)

mean_ic = daily_ic.dropna().mean()

In [ ]:
X_test

In [ ]:
mean_ic

In [ ]:
monthly_ic = daily_ic.groupby(daily_ic.index.to_period("M")).mean()
monthly_ic.plot(figsize=(10, 4))

In [ ]:
print("Mean IC:", daily_ic.mean())
print("Std IC :", daily_ic.std())
print("ICIR   :", daily_ic.mean() / daily_ic.std())
print("N days :", daily_ic.count())

In [ ]:
daily_ic_no_june = daily_ic[daily_ic.index < "2026-06-01"]

print("Mean IC:", daily_ic_no_june.mean())
print("Std IC :", daily_ic_no_june.std())
print("ICIR   :", daily_ic_no_june.mean() / daily_ic_no_june.std())
print("N days :", daily_ic_no_june.count())

In [ ]:
import shap
import numpy as np

explainer = shap.TreeExplainer(model.model)
shap_values = explainer.shap_values(X_test)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test,
    max_display=40,
    show=False
)

In [ ]:
mean_shap = np.abs(shap_values).mean(axis=0)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "mean_abs_shap": mean_shap
}).sort_values("mean_abs_shap", ascending=False)

importance.head(40)

In [ ]:
LightGBMTuner(20, 42, feature_matrix_market).run_data()

In [ ]:
shap.summary_plot(
    shap_values,
    X_test,
    max_display=40
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.lineplot(test_preds[test_preds["Ticker"] == "CBA.AX"], x= "Date", y="future_return_5d")
sns.lineplot(test_preds[test_preds["Ticker"] == "CBA.AX"], x= "Date", y="prediction")

In [ ]:
mean_ic

In [ ]:
test_preds

In [ ]:
test_preds

In [ ]:
test_preds

In [ ]:
test_preds

In [ ]:
test_preds.tail(20)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.lineplot(test_preds[test_preds["Ticker"] == "CBA.AX"], x= "Date", y="future_return_5d")
sns.lineplot(test_preds[test_preds["Ticker"] == "CBA.AX"], x= "Date", y="prediction")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.lineplot(test_preds[test_preds["Ticker"] == "ASX.AX"], x= "Date", y="future_return_5d")
sns.lineplot(test_preds[test_preds["Ticker"] == "ASX.AX"], x= "Date", y="prediction")

In [ ]:
tmp = test_preds.sort_values("Date").copy()

daily_avg = (
    tmp.groupby("Date")[["future_return_5d", "prediction"]]
       .mean()
       .reset_index()
)

daily_avg["future_return_5d_ma63"] = (
    daily_avg["future_return_5d"]
    .rolling(63, min_periods=10)
    .mean()
)

daily_avg["prediction_ma63"] = (
    daily_avg["prediction"]
    .rolling(63, min_periods=10)
    .mean()
)

daily_avg.plot(
    x="Date",
    y=["future_return_5d_ma63", "prediction_ma63"],
    figsize=(10, 4)
)

In [ ]:
ticker = "CBA.AX"   # change this

tmp = (
    test_preds[test_preds["Ticker"] == ticker]
    .sort_values("Date")
    .copy()
)

tmp[["future_return_5d", "prediction"]].describe()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(tmp["Date"], tmp["future_return_5d"], label="Actual 5d return")
ax.plot(tmp["Date"], tmp["prediction"], label="Prediction")

ax.axhline(0, linestyle="--")
ax.set_title(f"{ticker}: Actual vs Predicted 5-Day Return")
ax.set_xlabel("Date")
ax.set_ylabel("Return")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
tmp["actual_ma21"] = tmp["future_return_5d"].rolling(21, min_periods=5).mean()
tmp["pred_ma21"] = tmp["prediction"].rolling(21, min_periods=5).mean()

tmp.plot(
    x="Date",
    y=["actual_ma21", "pred_ma21"],
    figsize=(10, 4),
    title=f"{ticker}: Rolling Actual vs Prediction"
)

In [ ]:
test_preds[test_preds["Ticker"] == "CBA.AX"].tail(20)

In [ ]:
mean_ic

In [ ]:
import shap
import numpy as np

explainer = shap.TreeExplainer(model.model)
shap_values = explainer.shap_values(X_test)

In [ ]:
shap_importance = (
    pd.DataFrame({
        "feature": X_test.columns,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0)
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
shap_importance

In [ ]:
feature_matrix_subset = feature_matrix_market[["Date", "Ticker"] + shap_importance["feature"].tolist()[:10] + ["future_return_5d"]]

In [ ]:
LightGBMTuner(20, 42, feature_matrix_subset).run_data()

In [ ]:
shap.summary_plot(
    shap_values,
    X_test,
    max_display=40
)

In [ ]:
test_preds

In [ ]:
daily_ic.describe()

In [ ]:
0.014099/0.264


In [ ]:
positive_rate = (daily_ic > 0).mean()
print(positive_rate)

In [ ]:
bad_dates = daily_ic.sort_values().head(10).index

bad_preds = test_preds[test_preds["Date"].isin(bad_dates)].copy()

bad_summary = (
    bad_preds
    .groupby("Date")
    .agg(
        n_stocks=("Ticker", "count"),
        avg_target=("future_return_5d", "mean"),
        avg_prediction=("prediction", "mean"),
        std_target=("future_return_5d", "std"),
        std_prediction=("prediction", "std"),
    )
)

bad_summary

In [ ]:
test_preds["prediction"].describe()

In [ ]:
date = test_preds["Date"].iloc[0]

tmp = test_preds[test_preds["Date"] == date]

print(tmp["prediction"].describe())
print(tmp["future_return_5d"].describe())

In [ ]:
date = test_preds["Date"].iloc[0]

tmp = test_preds[test_preds["Date"] == date].copy()

tmp[["prediction", "future_return_5d"]].corr(method="spearman")

In [ ]:
bad_date = "2026-06-02"

tmp = test_preds[test_preds["Date"] == bad_date].copy()

tmp.sort_values("prediction", ascending=False)[
    ["Ticker", "prediction", "future_return_5d"]
].head(20)

tmp.sort_values("prediction", ascending=True)[
    ["Ticker", "prediction", "future_return_5d"]
].head(20)

In [ ]:
best_params

In [ ]:
directional_accuracy = (
    (output_test["prediction"] > 0)
    ==
    (output_test["future_return_5d"] > 0)
).mean()

In [ ]:
directional_accuracy

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(
    output_test["prediction"],
    output_test["future_return_5d"],
    alpha=0.2
)
plt.xlabel("Prediction")
plt.ylabel("Future Return 5D")

In [ ]:
output_test["decile"] = pd.qcut(
    output_test["prediction"],
    10,
    labels=False,
    duplicates="drop"
)

output_test.groupby("decile")["future_return_5d"].mean()

In [ ]:
output_test["future_return_5d"].corr(output_test["prediction"])

In [ ]:
daily_ic = (
    output_test
    .groupby("Date")
    .apply(lambda x: x["prediction"].corr(x["future_return_5d"]))
)

In [ ]:
print(daily_ic.mean())
print(daily_ic.std())
print((daily_ic > 0).mean())

In [ ]:
sns.scatterplot(data=output_test, x=)